# 13 Gold Patient Summary

## Purpose

This notebook creates a patient-level Gold analytics table.

## What We Are Doing

We will combine multiple Silver tables to create one summarized row per patient.

## Why We Are Doing This

Gold tables are business-ready tables used for:
- Power BI dashboards
- SQL analytics
- population health reporting
- ML feature engineering
- executive healthcare KPIs

## Final Output

`healthcare_catalog.gold.patient_summary`

## Final Grain

One row per patient.

## Step 1 — Import PySpark Functions

### What We Are Doing
We are importing Spark SQL functions.

### Why We Are Doing This
Gold tables require joins, aggregations, feature creation, and KPI calculations.

### Expected Output
Spark functions are available.

In [0]:
from pyspark.sql.functions import *

## Step 2 — Read Required Silver Tables

### What We Are Doing
We are reading clean Silver tables.

### Why We Are Doing This
Gold tables are created from trusted Silver tables, not raw Bronze data.

### Expected Output
DataFrames for patient, encounter, condition, medication, and claim data.

In [0]:
patient_df = spark.table("healthcare_catalog.silver.patient_clean")
encounter_df = spark.table("healthcare_catalog.silver.encounter_clean")
condition_df = spark.table("healthcare_catalog.silver.condition_clean")
medication_df = spark.table("healthcare_catalog.silver.medication_request_clean")
claim_df = spark.table("healthcare_catalog.silver.claim_clean")

print("Required Silver tables loaded successfully.")

Required Silver tables loaded successfully.


## Step 3 — Create Patient Demographic Base

### What We Are Doing
We are creating the patient-level base table.

### Why We Are Doing This
Every Gold patient-level table should start with one row per patient.

### Expected Output
A base DataFrame with patient demographics and age.

In [0]:
patient_base_df = patient_df.select(
    col("patient_id"),
    col("gender"),
    col("birth_date"),
    col("city"),
    col("state"),
    col("country"),
    col("postal_code"),
    col("marital_status")
).withColumn(
    "birth_date",
    to_date(col("birth_date"))
).withColumn(
    "age",
    floor(months_between(current_date(), col("birth_date")) / 12)
)

display(patient_base_df)

patient_id,gender,birth_date,city,state,country,postal_code,marital_status,age
b0a06ead-cc42-aa48-dad6-841d4aa679fa,male,1952-12-05,Boston,MA,US,02134,M,73
ccfc4db2-2026-7adb-3db0-33f3828140bb,male,1976-06-06,Peabody,MA,US,01940,M,49
31a2e8ec-69fc-8a71-3ab6-36cbdd508713,female,1917-05-15,Quincy,MA,US,02169,M,108
71a8b156-760b-df6b-859e-eefc7932a526,female,2014-11-15,Boston,MA,US,02120,Never Married,11
76b289fd-e825-734c-8446-316f59643593,female,1985-03-03,Boston,MA,US,02115,M,41
92fb7efc-5cfd-f8d3-927b-42f8ee099531,male,2013-06-12,Worcester,MA,US,01604,Never Married,12
81aa7647-779f-fd6b-94cf-782e606efeb2,female,2014-12-29,Weston,MA,US,null,Never Married,11
346a1435-2455-914f-c287-7b88052d05db,female,1981-11-16,Framingham,MA,US,01702,M,44
1cfa5a70-7f3c-4227-5cf1-e182fcff4cd4,female,1958-09-18,Quincy,MA,US,02186,M,67
97899f1d-9c3b-2b90-17b8-400c11ab8f0f,female,1969-10-19,Woburn,MA,US,01890,S,56


## Step 4 — Create Encounter Aggregates

### What We Are Doing
We are aggregating encounter utilization by patient.

### Why We Are Doing This
Encounter frequency is one of the most important healthcare utilization KPIs.

### Expected Output
One row per patient with encounter utilization metrics.

In [0]:
encounter_summary_df = encounter_df.groupBy(
    "patient_id"
).agg(
    count("*").alias("total_encounters"),

    sum(
        when(col("encounter_class") == "AMB", 1).otherwise(0)
    ).alias("ambulatory_encounters"),

    sum(
        when(col("encounter_class") == "EMER", 1).otherwise(0)
    ).alias("emergency_encounters"),

    sum(
        when(col("encounter_class") == "IMP", 1).otherwise(0)
    ).alias("inpatient_encounters")
)

display(encounter_summary_df)

patient_id,total_encounters,ambulatory_encounters,emergency_encounters,inpatient_encounters
b0a06ead-cc42-aa48-dad6-841d4aa679fa,32,30,2,0
ccfc4db2-2026-7adb-3db0-33f3828140bb,38,37,1,0
31a2e8ec-69fc-8a71-3ab6-36cbdd508713,64,63,0,1
71a8b156-760b-df6b-859e-eefc7932a526,19,18,1,0
76b289fd-e825-734c-8446-316f59643593,24,21,2,1
92fb7efc-5cfd-f8d3-927b-42f8ee099531,27,25,2,0
81aa7647-779f-fd6b-94cf-782e606efeb2,17,17,0,0
346a1435-2455-914f-c287-7b88052d05db,69,65,4,0
1cfa5a70-7f3c-4227-5cf1-e182fcff4cd4,38,37,1,0
97899f1d-9c3b-2b90-17b8-400c11ab8f0f,54,48,6,0


## Step 5 — Create Condition Aggregates

### What We Are Doing
We are counting conditions per patient.

### Why We Are Doing This
Condition count is a simple disease burden indicator.

### Expected Output
One row per patient with condition burden metrics.

In [0]:
condition_summary_df = condition_df.groupBy(
    "patient_id"
).agg(
    count("*").alias("total_conditions"),
    countDistinct("condition_description").alias("unique_conditions")
)

display(condition_summary_df)

patient_id,total_conditions,unique_conditions
b0a06ead-cc42-aa48-dad6-841d4aa679fa,38,16
ccfc4db2-2026-7adb-3db0-33f3828140bb,38,15
31a2e8ec-69fc-8a71-3ab6-36cbdd508713,78,15
71a8b156-760b-df6b-859e-eefc7932a526,3,3
76b289fd-e825-734c-8446-316f59643593,30,24
92fb7efc-5cfd-f8d3-927b-42f8ee099531,10,9
81aa7647-779f-fd6b-94cf-782e606efeb2,1,1
346a1435-2455-914f-c287-7b88052d05db,33,15
1cfa5a70-7f3c-4227-5cf1-e182fcff4cd4,53,22
97899f1d-9c3b-2b90-17b8-400c11ab8f0f,33,15


## Step 6 — Create Medication Aggregates

### What We Are Doing
We are counting medication requests per patient.

### Why We Are Doing This
Medication count is a proxy for treatment burden and clinical complexity.

### Expected Output
One row per patient with medication burden metrics.

In [0]:
medication_summary_df = medication_df.groupBy(
    "patient_id"
).agg(
    count("*").alias("total_medication_requests"),
    countDistinct("medication_name").alias("unique_medications")
)

display(medication_summary_df)

patient_id,total_medication_requests,unique_medications
b0a06ead-cc42-aa48-dad6-841d4aa679fa,53,7
ccfc4db2-2026-7adb-3db0-33f3828140bb,49,4
31a2e8ec-69fc-8a71-3ab6-36cbdd508713,1,1
71a8b156-760b-df6b-859e-eefc7932a526,4,3
76b289fd-e825-734c-8446-316f59643593,12,7
92fb7efc-5cfd-f8d3-927b-42f8ee099531,7,5
81aa7647-779f-fd6b-94cf-782e606efeb2,1,1
346a1435-2455-914f-c287-7b88052d05db,57,8
1cfa5a70-7f3c-4227-5cf1-e182fcff4cd4,26,4
97899f1d-9c3b-2b90-17b8-400c11ab8f0f,32,8


## Step 7 — Create Claim Cost Aggregates

### What We Are Doing
We are aggregating claim costs by patient.

### Why We Are Doing This
Claim cost is the main healthcare financial utilization metric.

### Expected Output
One row per patient with total healthcare cost.

In [0]:
claim_summary_df = claim_df.groupBy(
    "patient_id"
).agg(
    count("*").alias("total_claims"),
    sum("total_claim_amount").alias("total_claim_cost"),
    avg("total_claim_amount").alias("avg_claim_cost"),
    max("total_claim_amount").alias("max_claim_cost")
)

display(claim_summary_df)

patient_id,total_claims,total_claim_cost,avg_claim_cost,max_claim_cost
b0a06ead-cc42-aa48-dad6-841d4aa679fa,85,120811.62000000001,1421.3131764705884,89974.97
ccfc4db2-2026-7adb-3db0-33f3828140bb,87,40134.75000000001,461.31896551724145,1967.13
31a2e8ec-69fc-8a71-3ab6-36cbdd508713,65,208955.26999999973,3214.6964615384572,64760.45
71a8b156-760b-df6b-859e-eefc7932a526,23,22147.81000000001,962.9482608695656,10551.46
76b289fd-e825-734c-8446-316f59643593,36,126532.93000000005,3514.8036111111123,60259.75000000001
92fb7efc-5cfd-f8d3-927b-42f8ee099531,34,15155.11,445.7385294117647,1448.57
81aa7647-779f-fd6b-94cf-782e606efeb2,18,10938.01,607.6672222222222,1177.28
346a1435-2455-914f-c287-7b88052d05db,126,605520.1200000002,4805.71523809524,69511.58999999998
1cfa5a70-7f3c-4227-5cf1-e182fcff4cd4,64,86857.63000000002,1357.1504687500003,18137.41
97899f1d-9c3b-2b90-17b8-400c11ab8f0f,86,317028.87999999983,3686.382325581393,66527.89000000001


## Step 8 — Join All Patient-Level Summaries

### What We Are Doing
We are joining demographics, utilization, disease burden, medication burden, and cost metrics.

### Why We Are Doing This
This creates one enterprise patient-level analytic table.

### Expected Output
One row per patient with combined healthcare KPIs.

In [0]:
patient_summary_df = patient_base_df \
    .join(encounter_summary_df, on="patient_id", how="left") \
    .join(condition_summary_df, on="patient_id", how="left") \
    .join(medication_summary_df, on="patient_id", how="left") \
    .join(claim_summary_df, on="patient_id", how="left")

patient_summary_df = patient_summary_df.fillna(
    {
        "total_encounters": 0,
        "ambulatory_encounters": 0,
        "emergency_encounters": 0,
        "inpatient_encounters": 0,
        "total_conditions": 0,
        "unique_conditions": 0,
        "total_medication_requests": 0,
        "unique_medications": 0,
        "total_claims": 0,
        "total_claim_cost": 0,
        "avg_claim_cost": 0,
        "max_claim_cost": 0
    }
)

display(patient_summary_df)

patient_id,gender,birth_date,city,state,country,postal_code,marital_status,age,total_encounters,ambulatory_encounters,emergency_encounters,inpatient_encounters,total_conditions,unique_conditions,total_medication_requests,unique_medications,total_claims,total_claim_cost,avg_claim_cost,max_claim_cost
b0a06ead-cc42-aa48-dad6-841d4aa679fa,male,1952-12-05,Boston,MA,US,02134,M,73,32,30,2,0,38,16,53,7,85,120811.62000000001,1421.3131764705884,89974.97
ccfc4db2-2026-7adb-3db0-33f3828140bb,male,1976-06-06,Peabody,MA,US,01940,M,49,38,37,1,0,38,15,49,4,87,40134.75000000001,461.31896551724145,1967.13
31a2e8ec-69fc-8a71-3ab6-36cbdd508713,female,1917-05-15,Quincy,MA,US,02169,M,108,64,63,0,1,78,15,1,1,65,208955.26999999973,3214.6964615384572,64760.45
71a8b156-760b-df6b-859e-eefc7932a526,female,2014-11-15,Boston,MA,US,02120,Never Married,11,19,18,1,0,3,3,4,3,23,22147.81000000001,962.9482608695656,10551.46
76b289fd-e825-734c-8446-316f59643593,female,1985-03-03,Boston,MA,US,02115,M,41,24,21,2,1,30,24,12,7,36,126532.93000000005,3514.8036111111123,60259.75000000001
92fb7efc-5cfd-f8d3-927b-42f8ee099531,male,2013-06-12,Worcester,MA,US,01604,Never Married,12,27,25,2,0,10,9,7,5,34,15155.11,445.7385294117647,1448.57
81aa7647-779f-fd6b-94cf-782e606efeb2,female,2014-12-29,Weston,MA,US,null,Never Married,11,17,17,0,0,1,1,1,1,18,10938.01,607.6672222222222,1177.28
346a1435-2455-914f-c287-7b88052d05db,female,1981-11-16,Framingham,MA,US,01702,M,44,69,65,4,0,33,15,57,8,126,605520.1200000002,4805.71523809524,69511.58999999998
1cfa5a70-7f3c-4227-5cf1-e182fcff4cd4,female,1958-09-18,Quincy,MA,US,02186,M,67,38,37,1,0,53,22,26,4,64,86857.63000000002,1357.1504687500003,18137.41
97899f1d-9c3b-2b90-17b8-400c11ab8f0f,female,1969-10-19,Woburn,MA,US,01890,S,56,54,48,6,0,33,15,32,8,86,317028.87999999983,3686.382325581393,66527.89000000001


## Step 9 — Add Business Risk Flags

### What We Are Doing
We are creating simple healthcare risk flags.

### Why We Are Doing This
Gold tables should include business-friendly indicators for dashboards and ML.

### Expected Output
New risk flag columns.

In [0]:
patient_summary_df = patient_summary_df.withColumn(
    "has_emergency_visit_flag",
    when(col("emergency_encounters") > 0, 1).otherwise(0)
).withColumn(
    "has_inpatient_visit_flag",
    when(col("inpatient_encounters") > 0, 1).otherwise(0)
).withColumn(
    "high_utilization_flag",
    when(col("total_encounters") >= 50, 1).otherwise(0)
).withColumn(
    "high_cost_flag",
    when(col("total_claim_cost") >= 100000, 1).otherwise(0)
).withColumn(
    "high_disease_burden_flag",
    when(col("unique_conditions") >= 5, 1).otherwise(0)
)

display(patient_summary_df)

patient_id,gender,birth_date,city,state,country,postal_code,marital_status,age,total_encounters,ambulatory_encounters,emergency_encounters,inpatient_encounters,total_conditions,unique_conditions,total_medication_requests,unique_medications,total_claims,total_claim_cost,avg_claim_cost,max_claim_cost,has_emergency_visit_flag,has_inpatient_visit_flag,high_utilization_flag,high_cost_flag,high_disease_burden_flag
b0a06ead-cc42-aa48-dad6-841d4aa679fa,male,1952-12-05,Boston,MA,US,02134,M,73,32,30,2,0,38,16,53,7,85,120811.62000000001,1421.3131764705884,89974.97,1,0,0,1,1
ccfc4db2-2026-7adb-3db0-33f3828140bb,male,1976-06-06,Peabody,MA,US,01940,M,49,38,37,1,0,38,15,49,4,87,40134.75000000001,461.31896551724145,1967.13,1,0,0,0,1
31a2e8ec-69fc-8a71-3ab6-36cbdd508713,female,1917-05-15,Quincy,MA,US,02169,M,108,64,63,0,1,78,15,1,1,65,208955.26999999973,3214.6964615384572,64760.45,0,1,1,1,1
71a8b156-760b-df6b-859e-eefc7932a526,female,2014-11-15,Boston,MA,US,02120,Never Married,11,19,18,1,0,3,3,4,3,23,22147.81000000001,962.9482608695656,10551.46,1,0,0,0,0
76b289fd-e825-734c-8446-316f59643593,female,1985-03-03,Boston,MA,US,02115,M,41,24,21,2,1,30,24,12,7,36,126532.93000000005,3514.8036111111123,60259.75000000001,1,1,0,1,1
92fb7efc-5cfd-f8d3-927b-42f8ee099531,male,2013-06-12,Worcester,MA,US,01604,Never Married,12,27,25,2,0,10,9,7,5,34,15155.11,445.7385294117647,1448.57,1,0,0,0,1
81aa7647-779f-fd6b-94cf-782e606efeb2,female,2014-12-29,Weston,MA,US,null,Never Married,11,17,17,0,0,1,1,1,1,18,10938.01,607.6672222222222,1177.28,0,0,0,0,0
346a1435-2455-914f-c287-7b88052d05db,female,1981-11-16,Framingham,MA,US,01702,M,44,69,65,4,0,33,15,57,8,126,605520.1200000002,4805.71523809524,69511.58999999998,1,0,1,1,1
1cfa5a70-7f3c-4227-5cf1-e182fcff4cd4,female,1958-09-18,Quincy,MA,US,02186,M,67,38,37,1,0,53,22,26,4,64,86857.63000000002,1357.1504687500003,18137.41,1,0,0,0,1
97899f1d-9c3b-2b90-17b8-400c11ab8f0f,female,1969-10-19,Woburn,MA,US,01890,S,56,54,48,6,0,33,15,32,8,86,317028.87999999983,3686.382325581393,66527.89000000001,1,0,1,1,1


## Step 10 — Validate Final Patient Summary

### What We Are Doing
We are validating row count and key KPIs.

### Why We Are Doing This
The Gold patient summary should have one row per patient.

### Expected Output
555 rows, because our full dataset has 555 patients.

In [0]:
print("Patient Summary Row Count:", patient_summary_df.count())
print("Distinct Patient Count:", patient_summary_df.select("patient_id").distinct().count())

display(
    patient_summary_df.select(
        avg("age").alias("avg_age"),
        avg("total_encounters").alias("avg_encounters_per_patient"),
        avg("total_conditions").alias("avg_conditions_per_patient"),
        avg("total_claim_cost").alias("avg_claim_cost_per_patient")
    )
)

Patient Summary Row Count: 555
Distinct Patient Count: 555


avg_age,avg_encounters_per_patient,avg_conditions_per_patient,avg_claim_cost_per_patient
46.185585585585585,50.11171171171171,31.086486486486486,240605.0620180174


## Step 11 — Save Gold Patient Summary Table

### What We Are Doing
We are saving the final patient-level summary table into the Gold layer.

### Why We Are Doing This
This table becomes a reusable source for:
- SQL analytics
- Power BI dashboards
- ML feature engineering
- population health reporting

### Expected Output
A Delta table:

`healthcare_catalog.gold.patient_summary`

In [0]:
patient_summary_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("healthcare_catalog.gold.patient_summary")

print("Gold patient_summary table saved successfully.")

Gold patient_summary table saved successfully.


## Step 12 — Verify Gold Tables

### What We Are Doing
We are listing Gold tables.

### Why We Are Doing This
We want to confirm that `patient_summary` was successfully saved.

### Expected Output
`patient_summary` should appear in the Gold schema.

In [0]:
spark.sql("""
SHOW TABLES IN healthcare_catalog.gold
""").show(truncate=False)

+--------+---------------+-----------+
|database|tableName      |isTemporary|
+--------+---------------+-----------+
|gold    |patient_summary|false      |
+--------+---------------+-----------+

